# Step1A: QC Audit

Advanced notebook template for project-style single-cell QC.

This notebook focuses on the QC benchmark module:

- data loading and sample metadata checks
- QC metrics and cell-cycle scoring
- intelligent QC recommendation
- doublet detection
- sample-aware/adaptive marking
- filtering and retention audit
- QC report, review summary, module maturity, and compact audit output

Input: raw 10X directories or project h5ad configured in the parameter cell.  
Output: `data/processed/Step1-sce_cleaned.h5ad`.


## Environment Setup

In [ ]:
import gc
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import scLucid as scl

warnings.filterwarnings('ignore')

scl.setup_logging('INFO')
scl.set_figure_params(
    dpi=150,
    dpi_save=300,
    figsize=(6, 5),
    style='seaborn-v0_8',
    style_dict={'axes.grid': True},
    color_theme='default',
)


## Parameter Configuration**Edit this cell when switching projects.**All paths, sample names, metadata, and analysis options are centralized here.

In [ ]:
# ==============================================================================
# PROJECT PATHS
# ==============================================================================
# Set BASE_DIR to your project root. All other paths are relative to it.
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = BASE_DIR / 'data/RAW'            # 10X output directories (one per sample)
DATA_DIR = BASE_DIR / 'data/processed'     # Where intermediate .h5ad files are saved
RESULTS_DIR = BASE_DIR / 'results'         # All QC/preprocessing outputs go here
SAMPLE_METADATA_PATH = BASE_DIR / 'sample_metadata.csv'
SOURCE_NOTEBOOK_NAME = 'Step1-QC_and_Preprocessing.ipynb'
WORKFLOW_LAYER = 'advanced_notebook_api'

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# SPECIES & TISSUE
# ==============================================================================
SPECIES = 'mouse'               # 'human' or 'mouse'
TISSUE = None                   # Tissue context hint, e.g. 'tumor', 'blood', 'brain'
TISSUE_TYPE = 'tumor_tissue'    # 'tumor_tissue', 'pbmc_or_blood', 'cell_line', 'organoid', 'unknown'

# ==============================================================================
# SAMPLES & METADATA
# ==============================================================================
# Preferred: provide sample_metadata.csv with at least SAMPLE_KEY.
# Optional columns such as GROUP_KEY and BATCH_KEY are attached to adata.obs.
# If you do not use a metadata CSV, set ALL_SAMPLES explicitly and optionally fill GROUP_DICT/BATCH_DICT.
SAMPLE_KEY = 'sampleID'
ALL_SAMPLES = None              # None = derive from sample_metadata.csv; otherwise ['sample_1', 'sample_2']
GROUP_KEY = 'group'             # Set to None if no biological group metadata is available.
BATCH_KEY = 'batch'             # Set to None if no technical batch metadata is available.
GROUP_DICT = {}                 # Optional fallback: {'sample_1': 'control'}
BATCH_DICT = {}                 # Optional fallback: {'sample_1': 'batch_1'}
BIOLOGY_COLUMNS = [GROUP_KEY] if GROUP_KEY else []

# Optional custom colors. Leave empty for automatic Scanpy palettes.
SAMPLE_COLORS = {}
GROUP_COLORS = {}
BATCH_COLORS = {}

# ==============================================================================
# QC OPTIONS
# ==============================================================================
DOUBLET_METHOD = 'doubletdetection'  # 'scrublet', 'doubletdetection', or 'solo'
THRESHOLD_METHOD = 'mad'             # 'mad', 'gmm', or 'percentile'
MAD_MULTIPLIERS = [3, 4, 5]
MAX_CELLS_FOR_PLOTTING = 100_000

# Adaptive marking
ADAPTIVE_BATCH_KEY = SAMPLE_KEY
ADAPTIVE_METRICS = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
ADAPTIVE_METHOD = 'hierarchical'

# Manual threshold overrides.
# Floors protect against overly permissive data-driven lower bounds; ceilings should be used sparingly.
MANUAL_MIN_GENES = 300
MANUAL_MIN_COUNTS = None
MANUAL_MAX_GENES = None
MANUAL_MAX_COUNTS = None
MANUAL_PC_MT = None
MANUAL_NMADS = 5.0
PC_TOP_GENES_THRESHOLD = None        # e.g. 50.0 to enable fixed pc_top_20_genes filtering

CRITERIA_TO_FILTER = [
    'outlier_min_genes',
    'outlier_qc_metrics',
    'outlier_n_genes_by_counts_adaptive',
    'outlier_total_counts_adaptive',
    'outlier_mt',
    # 'outlier_pct_counts_mt_adaptive',
    'predicted_doublet',
]
FILTER_COMBINATION = 'any'

# ==============================================================================
# PREPROCESSING OPTIONS
# ==============================================================================
RUN_BIOTYPE_ANNOTATION = True
BIOTYPE_REFERENCE_PATH = None
BIOTYPE_METHOD = 'reference'         # 'reference' uses scLucid bundled reference; 'custom' requires BIOTYPE_REFERENCE_PATH
KEEP_BIOTYPES = ['protein_coding']
FILTER_BIOTYPE_AFTER_RAW = True

NORM_METHOD = 'standard'
NORM_TARGET_SUM = 1e4
EXCLUDE_HIGHLY_EXPRESSED = True
USE_QUALITY_AWARE_NORM = False

HVG_N_TOP_GENES_SCANPY = 2500
HVG_N_TOP_GENES_CUSTOM = 1500
HVG_FLAVOR = 'seurat'
HVG_KEY_SCANPY = f'highly_variable_scanpy_{HVG_FLAVOR}'
HVG_KEY_CUSTOM = 'highly_variable_custom'
HVG_UNION_MODE = 'union'
HVG_MIN_N_SAMPLES = 2
RUN_HVG_STABILITY = True

VARS_TO_REGRESS = ['total_counts', 'pct_counts_mt', 'S_score', 'G2M_score']
REGRESS_IN_SCALE = False
SCALE_MAX_VALUE = 10
N_PCS_MAX = 50

N_NEIGHBORS_LIST = [15, 20, 25, 30]
N_PCS_LIST = [20, 25, 30, 35, 40, 45, 50]
N_JOBS = -1

RUN_INTEGRATION = 'auto'             # True, False, or 'auto'
INTEGRATION_METHOD = 'harmony'
INTEGRATION_BATCH_KEY = BATCH_KEY or SAMPLE_KEY
EVALUATE_INTEGRATION = True
RUN_EMBEDDING_OPTIMIZATION = True

# ==============================================================================
# CHECKPOINT, DIAGNOSTICS & REPORTING
# ==============================================================================
USE_CHECKPOINTS = False
CHECKPOINT_DIR = RESULTS_DIR / 'checkpoints'
SHOW_NORMALIZATION_DIAGNOSTICS = True
SHOW_SCALING_DIAGNOSTICS = True
SHOW_INTEGRATION_DIAGNOSTICS = True
GENERATE_HTML_REPORT = False
VALIDATE_STAGES = ['qc', 'preprocess']

print('Configuration loaded.')
print(f'  Project: {BASE_DIR}')
print(f'  Species: {SPECIES}')
print(f'  Tissue type: {TISSUE_TYPE}')
print(f'  Sample metadata: {SAMPLE_METADATA_PATH}')


## Helper FunctionsThese functions handle common operations: metadata merging, sample key resolution,integration auto-detection, and filtering audit.

In [ ]:
def load_sample_metadata():
    """Load optional sample metadata and normalize sample identifiers to strings."""
    if SAMPLE_METADATA_PATH and Path(SAMPLE_METADATA_PATH).exists():
        sep = '\t' if Path(SAMPLE_METADATA_PATH).suffix.lower() in {'.tsv', '.txt'} else ','
        meta = pd.read_csv(SAMPLE_METADATA_PATH, sep=sep)
        if SAMPLE_KEY not in meta.columns:
            raise ValueError(f"sample metadata must contain SAMPLE_KEY column: {SAMPLE_KEY!r}")
        meta = meta.copy()
        meta[SAMPLE_KEY] = meta[SAMPLE_KEY].astype(str)
        if meta[SAMPLE_KEY].duplicated().any():
            dup = meta.loc[meta[SAMPLE_KEY].duplicated(), SAMPLE_KEY].tolist()
            raise ValueError(f"sample metadata contains duplicate samples: {dup}")
        return meta
    return None


def resolve_samples(sample_metadata=None):
    """Resolve sample list from config or metadata."""
    if ALL_SAMPLES is not None:
        return [str(s) for s in ALL_SAMPLES]
    if sample_metadata is not None:
        return sample_metadata[SAMPLE_KEY].tolist()
    raise ValueError(
        'No samples configured. Provide sample_metadata.csv or set ALL_SAMPLES explicitly.'
    )


def build_metadata_dicts(sample_metadata=None):
    """Build metadata dicts for load_10x_data from metadata CSV and optional fallback dicts."""
    metadata_dicts = {}
    if sample_metadata is not None:
        for col in sample_metadata.columns:
            if col == SAMPLE_KEY:
                continue
            if sample_metadata[col].notna().any():
                metadata_dicts[col] = dict(zip(sample_metadata[SAMPLE_KEY], sample_metadata[col]))
    if GROUP_DICT and GROUP_KEY:
        metadata_dicts[GROUP_KEY] = {str(k): v for k, v in GROUP_DICT.items()}
    if BATCH_DICT and BATCH_KEY:
        metadata_dicts[BATCH_KEY] = {str(k): v for k, v in BATCH_DICT.items()}
    return metadata_dicts


def validate_project_config(samples, metadata_dicts):
    """Fail early on template configuration issues that otherwise cause silent provenance debt."""
    if not samples:
        raise ValueError('No samples resolved from configuration.')
    missing_dirs = [s for s in samples if not (RAW_DIR / s).exists()]
    if missing_dirs:
        print(f"WARNING: {len(missing_dirs)} sample directories were not found under RAW_DIR.")
        print(f"  First missing entries: {missing_dirs[:5]}")
    for required_key in [GROUP_KEY, BATCH_KEY]:
        if required_key and required_key in metadata_dicts:
            missing = [s for s in samples if s not in metadata_dicts[required_key]]
            if missing:
                raise ValueError(f"metadata for {required_key!r} is missing samples: {missing}")
    if INTEGRATION_BATCH_KEY and INTEGRATION_BATCH_KEY in BIOLOGY_COLUMNS:
        print(
            f"WARNING: INTEGRATION_BATCH_KEY={INTEGRATION_BATCH_KEY!r} is also listed as biology. "
            'Auto integration will likely skip correction to avoid removing biology.'
        )


def print_sample_crosstab(adata, group_key=None):
    """Print a crosstab of samples x groups for a quick data overview."""
    if group_key and group_key in adata.obs.columns:
        ctab = pd.crosstab(adata.obs[SAMPLE_KEY], adata.obs[group_key], margins=True)
    else:
        ctab = pd.DataFrame(adata.obs[SAMPLE_KEY].value_counts())
    print(ctab)


def audit_filtering(adata_before, adata_after, group_key=None):
    """Compare cell counts before and after filtering, by sample and optionally by group."""
    before = adata_before.obs.groupby(SAMPLE_KEY, observed=True).size()
    after = adata_after.obs.groupby(SAMPLE_KEY, observed=True).size()
    audit = pd.DataFrame({'before': before, 'after': after}).fillna(0).astype(int)
    audit['removed'] = audit['before'] - audit['after']
    audit['retention_rate'] = (audit['after'] / audit['before']).round(3)
    print('=== Retention by sample ===')
    display(audit.reset_index())

    audit_g = None
    if group_key and group_key in adata_before.obs.columns:
        before_g = adata_before.obs.groupby(group_key, observed=True).size()
        after_g = adata_after.obs.groupby(group_key, observed=True).size()
        audit_g = pd.DataFrame({'before': before_g, 'after': after_g}).fillna(0).astype(int)
        audit_g['removed'] = audit_g['before'] - audit_g['after']
        audit_g['retention_rate'] = (audit_g['after'] / audit_g['before']).round(3)
        print('=== Retention by group ===')
        display(audit_g.reset_index())
    return {'sample': audit, 'group': audit_g}


def audit_doublets(adata):
    """Check remaining doublet calls after filtering."""
    for col in ['predicted_doublet', 'scrublet_predicted', 'doubletdetection_predicted', 'heuristic_predicted']:
        if col in adata.obs.columns:
            n = adata.obs[col].sum()
            print(f'  {col}: {n} cells ({n / adata.n_obs * 100:.2f}%)')
            if col == 'predicted_doublet' and n > 0:
                print(f'    NOTE: {n} predicted doublets remain after filtering; review FilterConfig criteria.')


def detect_integration_confounding(adata, batch_key, biology_columns):
    """Check whether batch_key is confounded with biology columns."""
    if not batch_key or batch_key not in adata.obs.columns:
        return ['batch_key_missing']
    problems = []
    for col in biology_columns:
        if not col or col not in adata.obs.columns:
            continue
        mapping = adata.obs[[batch_key, col]].dropna().drop_duplicates()
        per_batch = mapping.groupby(batch_key)[col].nunique()
        if len(per_batch) > 0 and (per_batch <= 1).all():
            n_batch = adata.obs[batch_key].nunique()
            n_col = adata.obs[col].nunique()
            if n_batch == n_col:
                problems.append(f'{batch_key} is one-to-one with {col}; integration would remove biological signal')
    return problems


def resolve_integration_flag(adata, batch_key, biology_columns):
    """Determine whether to run integration, with auto-detection support."""
    if RUN_INTEGRATION is True:
        return True, []
    if RUN_INTEGRATION is False:
        return False, []
    if not batch_key or batch_key not in adata.obs.columns or adata.obs[batch_key].nunique() < 2:
        return False, ['single batch/sample; no integration needed']
    problems = detect_integration_confounding(adata, batch_key, biology_columns)
    return len(problems) == 0, problems


def build_color_palette(keys):
    """Build a combined color palette from configured label-color dictionaries."""
    palette = {}
    for k in keys:
        if k == SAMPLE_KEY:
            palette.update(SAMPLE_COLORS)
        elif GROUP_KEY and k == GROUP_KEY:
            palette.update(GROUP_COLORS)
        elif BATCH_KEY and k == BATCH_KEY:
            palette.update(BATCH_COLORS)
    return palette if palette else None


def build_config_snapshot():
    """Collect notebook parameters into a serializable config snapshot."""
    return {
        'project': {
            'base_dir': str(BASE_DIR),
            'raw_dir': str(RAW_DIR),
            'data_dir': str(DATA_DIR),
            'results_dir': str(RESULTS_DIR),
            'sample_metadata_path': str(SAMPLE_METADATA_PATH),
            'source_notebook': SOURCE_NOTEBOOK_NAME,
            'workflow_layer': WORKFLOW_LAYER,
        },
        'metadata': {
            'sample_key': SAMPLE_KEY,
            'group_key': GROUP_KEY,
            'batch_key': BATCH_KEY,
            'biology_columns': [c for c in BIOLOGY_COLUMNS if c],
        },
        'qc': {
            'doublet_method': DOUBLET_METHOD,
            'threshold_method': THRESHOLD_METHOD,
            'mad_multipliers': MAD_MULTIPLIERS,
            'adaptive_batch_key': ADAPTIVE_BATCH_KEY,
            'adaptive_metrics': ADAPTIVE_METRICS,
            'adaptive_method': ADAPTIVE_METHOD,
            'manual_min_genes': MANUAL_MIN_GENES,
            'manual_min_counts': MANUAL_MIN_COUNTS,
            'manual_max_genes': MANUAL_MAX_GENES,
            'manual_max_counts': MANUAL_MAX_COUNTS,
            'manual_pc_mt': MANUAL_PC_MT,
            'manual_nmads': MANUAL_NMADS,
            'pc_top_genes_threshold': PC_TOP_GENES_THRESHOLD,
            'criteria_to_filter': CRITERIA_TO_FILTER,
            'filter_combination': FILTER_COMBINATION,
            'tissue_type': TISSUE_TYPE,
            'species': SPECIES,
        },
        'preprocess': {
            'biotype_method': BIOTYPE_METHOD,
            'keep_biotypes': KEEP_BIOTYPES,
            'norm_method': NORM_METHOD,
            'norm_target_sum': NORM_TARGET_SUM,
            'exclude_highly_expressed': EXCLUDE_HIGHLY_EXPRESSED,
            'use_quality_aware_norm': USE_QUALITY_AWARE_NORM,
            'hvg_key_scanpy': HVG_KEY_SCANPY,
            'hvg_key_custom': HVG_KEY_CUSTOM,
            'hvg_union_mode': HVG_UNION_MODE,
            'vars_to_regress': VARS_TO_REGRESS,
            'n_pcs_max': N_PCS_MAX,
            'run_integration': RUN_INTEGRATION,
            'integration_method': INTEGRATION_METHOD,
            'integration_batch_key': INTEGRATION_BATCH_KEY,
        },
    }


def finalize_manual_review_summary(adata, module, workflow_name, steps, config, summary, save_dir=None):
    """Write the shared scLucid review contract for an advanced notebook workflow.

    The canonical review summary is a flat envelope matching workflow runs.
    `normalize_review_summary()` also adds a legacy `data` view so older
    notebooks/tests can read `review_summary['data']` during the schema
    transition.
    """
    from scLucid.utils import (
        build_config_lineage,
        export_review_summary,
        normalize_review_summary,
        record_config_lineage,
        save_result,
        save_workflow_result,
        validate_review_summary_schema,
    )

    inherited_context = {
        'source': 'notebook',
        'source_notebook': SOURCE_NOTEBOOK_NAME,
        'workflow_layer': WORKFLOW_LAYER,
        'species': SPECIES,
        'tissue': TISSUE,
        'tissue_type': TISSUE_TYPE,
        'sample_key': SAMPLE_KEY,
        'batch_key': BATCH_KEY,
        'group_key': GROUP_KEY,
    }
    lineage = build_config_lineage(
        inherited=inherited_context,
        stage_config=config,
        effective_config=config,
    )
    lineage['notebook'] = {
        'source_notebook': SOURCE_NOTEBOOK_NAME,
        'workflow_layer': WORKFLOW_LAYER,
        'config_snapshot': build_config_snapshot(),
    }
    review_summary = normalize_review_summary(
        summary,
        module=module,
        workflow_name=workflow_name,
        adata=adata,
        steps_executed=steps,
        config=config,
        config_lineage=lineage,
        artifacts={},
    )
    validate_review_summary_schema(review_summary, module=module, raise_on_error=True)
    save_result(adata, module, 'review_summary', review_summary)
    save_workflow_result(adata, module, workflow_name, steps, config)
    record_config_lineage(adata, module, lineage)
    if save_dir is not None:
        export_review_summary(review_summary, save_dir, module, adata=adata)
    return review_summary


print('Helper functions defined.')


---## **Step 1: Data Loading**Load 10X-format data using `scLucid.utils.load_10x_data()`.This function handles multi-sample loading, metadata attachment, and savesa combined raw `.h5ad` for reproducibility.If your data is already in `.h5ad` format, skip to the next section and use`sc.read_h5ad()`.

In [ ]:
# Build metadata dicts from configuration.
sample_metadata = load_sample_metadata()
ALL_SAMPLES = resolve_samples(sample_metadata)
metadata_dicts = build_metadata_dicts(sample_metadata)
validate_project_config(ALL_SAMPLES, metadata_dicts)

print(f'Resolved {len(ALL_SAMPLES)} samples: {ALL_SAMPLES}')
if sample_metadata is not None:
    display(sample_metadata)

# Load all samples.
adata = scl.utils.load_10x_data(
    samples=ALL_SAMPLES,
    base_dir=str(RAW_DIR),
    metadata_dicts=metadata_dicts,
    output_file=str(DATA_DIR / 'Step0-combined_raw_data.h5ad'),
)
print(f'Loaded {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print_sample_crosstab(adata, group_key=GROUP_KEY)
adata


---## **Step 2: Quality Control (QC)**### QC strategy overviewscLucid provides a layered QC approach:1. **Calculate metrics** — n_genes, n_counts, pct_mito, pct_ribo, cell cycle scores2. **Intelligent recommendations** — data-driven threshold suggestions with confidence intervals3. **Doublet detection** — algorithmic + heuristic marker-based, merged via ensemble4. **Threshold suggestion** — MAD-based or GMM-based global thresholds5. **Adaptive marking** — per-sample adaptive thresholds, catching sample-specific outliers6. **Unified marking** — combine all outlier flags + doublet calls into a single decision7. **Filter + audit** — apply filter and produce retention statistics8. **Report** — generate text and optional HTML QC reportEach step can be inspected before proceeding to the next.

In [ ]:
gc.collect()# Load the combined raw dataadata = sc.read_h5ad(str(DATA_DIR / "Step0-combined_raw_data.h5ad"))print(f"Dataset loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes")print_sample_crosstab(adata, group_key=GROUP_KEY)print(f"\n{'='*60}")print("Quality Control")print(f"{'='*60}")

### 2.1 Calculate QC Metrics & Cell CycleComputes per-cell quality metrics:- `n_genes_by_counts`, `total_counts` — library complexity and depth- `pct_counts_mt`, `pct_counts_ribo`, `pct_counts_hb` — contamination indicators- `S_score`, `G2M_score`, `phase` — cell cycle scoring (canonical markers)Violin plots and scatter plots are generated for visual inspection.

In [ ]:
reporting_config = scl.qc.MetricsReportingConfig(
    plot_scatter=False,
    save_dir=str(RESULTS_DIR / 'qc_metrics'),
)
adata = scl.qc.calculate_qc_metric(
    adata,
    sample_key=SAMPLE_KEY,
    calculate_cell_cycle=True,
    cell_cycle_species=SPECIES,
    max_cells_for_plotting=MAX_CELLS_FOR_PLOTTING,
    reporting_config=reporting_config,
)
print('QC metrics calculated.')
metric_cols = [
    c for c in adata.obs.columns
    if c.startswith(('n_', 'total_', 'pct_', 'S_score', 'G2M_score', 'phase'))
]
print(f'  Metrics in adata.obs: {metric_cols}')


### 2.2 Intelligent QC RecommendationsscLucid's **IntelligentQCRecommender** analyzes the distribution of QC metrics andproposes data-driven thresholds using multiple methods (GMM, bootstrapping, distribution fitting).This is a core innovation of scLucid — rather than using fixed cutoffs (e.g. "n_genes > 200"),the recommender inspects your data and provides thresholds with **confidence intervals**.The output includes:- A `QCRecommendation` object with threshold suggestions and confidence intervals- Diagnostic plots showing the underlying distribution fits- Per-sample recommendations when sample_key is provided

In [ ]:
from scLucid.qc import recommend_intelligent_qc

print("Running Intelligent QC Recommender...")
intelligent_recommendations = recommend_intelligent_qc(
    adata,
    tissue_type=TISSUE_TYPE,
    strategy="auto",
    plot=True,
    save_dir=str(RESULTS_DIR / "qc_intelligent"),
)

# Access recommendation attributes directly (dataclass, not methods)
rec = intelligent_recommendations
print(f"\nOverall confidence: {rec.overall_confidence:.2f}")
print(f"Data quality score: {rec.data_quality_score:.1f}/100")
print(f"Strategy: {rec.overall_strategy.value}")
print(f"\nRecommended thresholds (with 95% CI):")
print(f"  min_genes:      {rec.min_genes.threshold} "
      f"[{rec.min_genes.ci_lower}, {rec.min_genes.ci_upper}]")
print(f"  max_mt_percent: {rec.max_mt_percent.threshold} "
      f"[{rec.max_mt_percent.ci_lower}, {rec.max_mt_percent.ci_upper}]")
print(f"  n_counts:       {rec.n_counts.threshold} "
      f"[{rec.n_counts.ci_lower}, {rec.n_counts.ci_upper}]  (lower bound)")
print(f"  doublet_score:  {rec.doublet_threshold.threshold} "
      f"[{rec.doublet_threshold.ci_lower}, {rec.doublet_threshold.ci_upper}]")

if rec.concerns:
    print(f"\nConcerns:")
    for c in rec.concerns:
        print(f"  - {c}")

if rec.tumor_specific_considerations:
    print(f"\nTumor-specific considerations:")
    for c in rec.tumor_specific_considerations:
        print(f"  - {c}")

# Build thresholds dict for downstream use.
# IMPORTANT: rec.n_counts is a LOWER bound (15th percentile of log-normal fit).
# rec.max_mt_percent is an UPPER bound (distribution fitting).
rec_thresholds = {
    "min_genes":   rec.min_genes.threshold,         # lower bound
    "max_genes":   None,                            # not provided
    "min_counts":  rec.n_counts.threshold,          # lower bound — NOT a max!
    "max_counts":  None,                            # not provided; MAD fills this
    "pc_mt":       rec.max_mt_percent.threshold,    # upper bound
}
print(f"\nExtracted thresholds for downstream use:")
for k, v in rec_thresholds.items():
    print(f"  {k}: {v}")


### 2.3 Doublet DetectionscLucid supports multiple doublet detection methods:- **scrublet** — fast, heuristic-based (good default for most data)- **doubletdetection** — more sensitive, uses co-expression patterns- **solo** — deep learning based (requires scvi-tools)The recommended approach is to:1. Auto-calculate expected doublet rates from the number of recovered cells per sample2. Run algorithmic detection with the selected method3. Enable heuristic marker-based detection as an independent check4. Merge results via weighted averaging (`merge_strategy="weighted_average"`)

In [ ]:
print("---- Doublet Detection ----")# Auto-calculate expected doublet rates per sampledoublet_rates = scl.qc.generate_doublet_rates(adata, sample_key=SAMPLE_KEY)print(f"Expected doublet rates: {doublet_rates}")doublet_cfg = scl.qc.DoubletConfig(    run_algorithm=True,    method=DOUBLET_METHOD,    expected_doublet_rate=doublet_rates,    use_heuristics=True,    marker_species=SPECIES,    merge_strategy="weighted_average",    algorithm_weight=0.7,    plot_summary=True,    plot_bar=True,    plot_scatter=True,    plot_upset=False,    show_plots=True,    save_dir=str(RESULTS_DIR / "qc_doublets"),)adata = scl.qc.predict_doublets(    adata,    config=doublet_cfg,    sample_key=SAMPLE_KEY,)# Quick summaryfor col in ["scrublet_predicted", "doubletdetection_predicted", "heuristic_predicted", "predicted_doublet"]:    if col in adata.obs.columns:        n = adata.obs[col].sum()        print(f"  {col}: {n} cells ({n/adata.n_obs*100:.2f}%)")

### 2.4 Suggest QC ThresholdsComputes global reference thresholds using the specified method (MAD, GMM, or percentile).These thresholds serve as a reference; per-sample adaptive marking is applied in the next step.The MAD method identifies outliers based on median absolute deviation, which is robustto non-normal distributions common in scRNA-seq data.

In [ ]:
print("\n---- Suggesting QC Thresholds ----")

threshold_table, suggested_thresholds = scl.qc.suggest_qc_thresholds(
    adata,
    method=THRESHOLD_METHOD,
    mad_multipliers=MAD_MULTIPLIERS,
    plot_distributions=True,
    save_dir=str(RESULTS_DIR / "qc_metrics"),
)

print("Suggested global QC thresholds:")
display(threshold_table)
print(suggested_thresholds.to_dict())

# ── Merge three threshold sources ──
# Priority: IntelligentQCRecommender > MAD-based suggestion > hardcoded fallback
# MANUAL_MIN_GENES / MANUAL_MIN_COUNTS act as floors (not overrides).
# MANUAL_MAX_GENES / MANUAL_MAX_COUNTS / MANUAL_PC_MT act as ceilings when set.

# --- min_genes: data-driven with manual floor ---
smart_min_genes = (
    rec_thresholds.get("min_genes")
    or getattr(suggested_thresholds, 'min_genes', None)
    or 300
)
final_min_genes = max(MANUAL_MIN_GENES or 0, smart_min_genes)

# --- min_counts: data-driven, manual overrides if set ---
smart_min_counts = (
    rec_thresholds.get("min_counts")
    or getattr(suggested_thresholds, 'min_counts', None)
)
final_min_counts = (
    MANUAL_MIN_COUNTS if MANUAL_MIN_COUNTS is not None
    else smart_min_counts
)

# --- max_genes: manual ceiling if set, else data-driven ---
final_max_genes = (
    MANUAL_MAX_GENES if MANUAL_MAX_GENES is not None
    else rec_thresholds.get("max_genes")
    or getattr(suggested_thresholds, 'max_genes', None)
)

# --- max_counts: manual ceiling if set, else data-driven ---
final_max_counts = (
    MANUAL_MAX_COUNTS if MANUAL_MAX_COUNTS is not None
    else rec_thresholds.get("max_counts")
    or getattr(suggested_thresholds, 'max_counts', None)
)

# --- pc_mt: manual ceiling if set, else data-driven ---
final_pc_mt = (
    MANUAL_PC_MT if MANUAL_PC_MT is not None
    else rec_thresholds.get("pc_mt")
    or getattr(suggested_thresholds, 'pc_mt', None)
    or 10.0
)

print(f"\nFinal thresholds for cell marking:")
print(f"  min_genes  = {final_min_genes}   (smart={smart_min_genes}, floor={MANUAL_MIN_GENES})")
print(f"  min_counts = {final_min_counts}  (smart={smart_min_counts}, manual={MANUAL_MIN_COUNTS})")
print(f"  max_genes  = {final_max_genes}   (manual={MANUAL_MAX_GENES})")
print(f"  max_counts = {final_max_counts}  (manual={MANUAL_MAX_COUNTS})")
print(f"  pc_mt      = {final_pc_mt}       (manual={MANUAL_PC_MT})")


### 2.5 Mark Low-Quality Cells (Sample-Aware + Adaptive)Two-stage marking:1. **Adaptive** — per-sample outlier detection using hierarchical or independent method.   This catches cells that are outliers *within their own sample*, which is essential   for multi-sample data where overall distributions may differ.2. **Unified** — combines adaptive flags, global MAD-based flags, and doublet calls   into a comprehensive set of boolean columns in `adata.obs`.The columns_to_plot list should be updated based on which doublet method was run.

In [ ]:
print('\n---- Adaptive Cell Marking (sample-aware) ----')
adata = scl.qc.mark_low_quality_cells_adaptive(
    adata,
    batch_key=ADAPTIVE_BATCH_KEY,
    metrics=ADAPTIVE_METRICS,
    method=ADAPTIVE_METHOD,
)
print('Adaptive marking complete.')
adaptive_cols = [c for c in adata.obs.columns if '_adaptive' in c]
print(f'  Adaptive columns: {adaptive_cols}')


In [ ]:
print('\n---- Unified Cell Marking ----')

marking_cols_to_plot = [
    'outlier_min_genes',
    'outlier_max_genes',
    'outlier_qc_metrics',
    'outlier_n_genes_by_counts_adaptive',
    'outlier_total_counts_adaptive',
    'outlier_mt',
    'outlier_pct_counts_mt_adaptive',
    'predicted_doublet',
]
for dc in ['doubletdetection_predicted', 'scrublet_predicted', 'heuristic_predicted']:
    if dc in adata.obs.columns:
        marking_cols_to_plot.append(dc)

top_gene_thresholds = (
    {'pc_top_20_genes': PC_TOP_GENES_THRESHOLD}
    if PC_TOP_GENES_THRESHOLD is not None
    else {}
)

marking_cfg = scl.qc.MarkingConfig(
    thresholds=scl.qc.QCThresholds(
        min_genes=final_min_genes,
        max_genes=final_max_genes,
        min_counts=final_min_counts,
        max_counts=final_max_counts,
        pc_mt=final_pc_mt,
        pc_top_genes=top_gene_thresholds,
        use_fixed_top_gene_threshold=PC_TOP_GENES_THRESHOLD is not None,
        nmads=MANUAL_NMADS,
    ),
    cols_to_plot=marking_cols_to_plot,
    save_dir=str(RESULTS_DIR / 'qc_marking_plots'),
)

adata = scl.qc.mark_low_quality_cell(
    adata,
    sample_key=SAMPLE_KEY,
    config=marking_cfg,
)

for col in marking_cols_to_plot:
    if col in adata.obs.columns:
        n = adata.obs[col].sum() if adata.obs[col].dtype == bool else 'non-bool'
        if isinstance(n, (int, float)):
            print(f'  {col}: {n} cells flagged ({n / adata.n_obs * 100:.1f}%)')


### 2.6 Filter Cells & Retention AuditApplies the configured filter criteria and produces a detailed retention audit:- Per-sample cell retention before/after- Per-group retention (if group metadata available)- Doublet-specific audit to confirm all doublets are removed

In [ ]:
print('\n---- Filtering Cells ----')
filter_cfg = scl.qc.FilterConfig(
    criteria_to_filter=CRITERIA_TO_FILTER,
    combination_logic=FILTER_COMBINATION,
)
adata_filtered = scl.qc.filter_cells(
    adata,
    config=filter_cfg,
    copy=True,
)

filtering_audit = audit_filtering(adata, adata_filtered, group_key=GROUP_KEY)

print('\n=== Doublet Audit ===')
audit_doublets(adata_filtered)

remaining_doublets = int(
    adata_filtered.obs.get('predicted_doublet', pd.Series(False, index=adata_filtered.obs_names)).sum()
)
if remaining_doublets > 0:
    print(f'\nWARNING: {remaining_doublets} predicted_doublet=True cells remain after filtering!')
    print("Review CRITERIA_TO_FILTER to ensure 'predicted_doublet' is included.")


### 2.7 Generate QC ReportsTwo report formats are available:1. **Standard text/CSV report** — `generate_qc_report()` — always generated2. **Interactive HTML report** — `generate_qc_html_report()` — optional, requires plotlyThe HTML report includes embedded stat cards, interactive Plotly charts, anda complete record of QC decisions for reviewer traceability.

In [ ]:
print("\n---- Generating QC Reports ----")

# Standard report (always generated)
scl.qc.generate_qc_report(
    adata_filtered,
    save_dir=str(RESULTS_DIR / "qc_report"),
    sample_key=SAMPLE_KEY,
    adata_before=adata,
)

# Optional interactive HTML report
if GENERATE_HTML_REPORT:
    try:
        scl.qc.generate_qc_html_report(
            adata_filtered,
            output_path=str(RESULTS_DIR / "qc_html_report" / "qc_report.html"),
            adata_before=adata,
            title="scLucid Quality Control Report",
        )
        print("HTML QC report generated.")
    except ImportError:
        print("Skipping HTML report — plotly not available. Install with: pip install plotly")
    except Exception as e:
        print(f"HTML report generation failed: {e}")

print("\nQC reporting complete.")


### 2.8 QC Audit SummaryFinal summary of QC results before moving to preprocessing.

In [ ]:
print('\n---- QC Audit Summary ----')
print(f'Before QC:  {adata.n_obs:,} cells, {adata.n_vars:,} genes')
print(f'After QC:   {adata_filtered.n_obs:,} cells, {adata_filtered.n_vars:,} genes')
print(
    f'Removed:    {adata.n_obs - adata_filtered.n_obs:,} cells '
    f'({(1 - adata_filtered.n_obs / adata.n_obs) * 100:.1f}%)'
)
print_sample_crosstab(adata_filtered, group_key=GROUP_KEY)
audit_doublets(adata_filtered)


In [ ]:
qc_steps_executed = [
    'load_10x_data',
    'calculate_qc_metric',
    'recommend_intelligent_qc',
    'predict_doublets',
    'suggest_qc_thresholds',
    'mark_low_quality_cells_adaptive',
    'mark_low_quality_cell',
    'filter_cells',
    'generate_qc_report',
]

qc_warnings = []
if remaining_doublets > 0:
    qc_warnings.append(f'{remaining_doublets} predicted doublets remain after filtering')

qc_filtering_summary = {
    'initial_cells': int(adata.n_obs),
    'final_cells': int(adata_filtered.n_obs),
    'cells_before': int(adata.n_obs),
    'cells_after': int(adata_filtered.n_obs),
    'cells_removed': int(adata.n_obs - adata_filtered.n_obs),
    'retention_rate': float(adata_filtered.n_obs / adata.n_obs) if adata.n_obs else None,
    'criteria_to_filter': list(CRITERIA_TO_FILTER),
    'combination_logic': FILTER_COMBINATION,
}

qc_review_config = {
    'marking_config': {
        'thresholds': {
            'min_genes': final_min_genes,
            'min_counts': final_min_counts,
            'max_genes': final_max_genes,
            'max_counts': final_max_counts,
            'pc_mt': final_pc_mt,
            'nmads': MANUAL_NMADS,
        }
    },
    'doublet_config': {
        'method': DOUBLET_METHOD,
        'score_threshold': rec_thresholds.get('doublet_score') if 'rec_thresholds' in globals() else None,
    },
    'filter_config': {
        'criteria_to_filter': list(CRITERIA_TO_FILTER),
        'combination': FILTER_COMBINATION,
    },
}
qc_original_config = {
    'marking_config': {
        'thresholds': {
            'min_genes': MANUAL_MIN_GENES,
            'min_counts': MANUAL_MIN_COUNTS,
            'max_genes': MANUAL_MAX_GENES,
            'max_counts': MANUAL_MAX_COUNTS,
            'pc_mt': MANUAL_PC_MT,
            'nmads': MANUAL_NMADS,
        }
    },
    'doublet_config': {'method': DOUBLET_METHOD},
    'filter_config': {'criteria_to_filter': list(CRITERIA_TO_FILTER)},
}
qc_user_overrides = {
    key: value
    for key, value in {
        'min_genes': MANUAL_MIN_GENES,
        'min_counts': MANUAL_MIN_COUNTS,
        'max_genes': MANUAL_MAX_GENES,
        'max_counts': MANUAL_MAX_COUNTS,
        'max_mt_percent': MANUAL_PC_MT,
        'nmads': MANUAL_NMADS,
    }.items()
    if value is not None
}
qc_sample_thresholds = {
    'batch_key': ADAPTIVE_BATCH_KEY,
    'metrics': list(ADAPTIVE_METRICS),
    'method': ADAPTIVE_METHOD,
    'adaptive_columns': adaptive_cols if 'adaptive_cols' in globals() else [],
}
qc_context = {
    'sample_key': SAMPLE_KEY,
    'threshold_mode': THRESHOLD_METHOD,
    'n_samples': int(adata.obs[SAMPLE_KEY].nunique()) if SAMPLE_KEY in adata.obs else None,
    'tissue_type': TISSUE_TYPE,
    'use_recommendations': 'intelligent_recommendations' in globals(),
}

qc_summary = {
    'workflow_layer': WORKFLOW_LAYER,
    'recommendation_summary': {
        'available': 'intelligent_recommendations' in globals(),
        'overall_confidence': getattr(intelligent_recommendations, 'overall_confidence', None) if 'intelligent_recommendations' in globals() else None,
        'data_quality_score': getattr(intelligent_recommendations, 'data_quality_score', None) if 'intelligent_recommendations' in globals() else None,
        'strategy': str(getattr(getattr(intelligent_recommendations, 'overall_strategy', None), 'value', None)) if 'intelligent_recommendations' in globals() else None,
    },
    'threshold_summary': {
        'min_genes': final_min_genes,
        'min_counts': final_min_counts,
        'max_genes': final_max_genes,
        'max_counts': final_max_counts,
        'pc_mt': final_pc_mt,
        'manual_min_genes_floor': MANUAL_MIN_GENES,
        'manual_pc_mt_ceiling': MANUAL_PC_MT,
        'pc_top_genes_threshold': PC_TOP_GENES_THRESHOLD,
    },
    'applied_threshold_summary': qc_review_config['marking_config']['thresholds'],
    'user_override_summary': {'details': qc_user_overrides},
    'sample_threshold_summary': qc_sample_thresholds,
    'tumor_aware_summary': {
        'enabled': 'tumor' in str(TISSUE_TYPE).lower() or 'cancer' in str(TISSUE_TYPE).lower(),
        'tissue_type': TISSUE_TYPE,
        'note': 'Tumor-aware QC: mitochondrial and stress signals should be reviewed before hard filtering.' if 'tumor' in str(TISSUE_TYPE).lower() else None,
    },
    'filtering_summary': qc_filtering_summary,
    'sample_retention': filtering_audit['sample'].reset_index().to_dict(orient='records')
    if 'filtering_audit' in globals() else [],
    'warnings': qc_warnings,
}

from scLucid.qc.trace import enrich_qc_review_summary, validate_qc_review_summary

qc_summary = enrich_qc_review_summary(
    qc_summary,
    adata=adata_filtered,
    config=qc_review_config,
    original_config=qc_original_config,
    recommendation=intelligent_recommendations if 'intelligent_recommendations' in globals() else None,
    sample_thresholds=qc_sample_thresholds,
    filtering_summary=qc_filtering_summary,
    warnings=qc_warnings,
    context=qc_context,
    steps_executed=qc_steps_executed,
)

qc_review_summary = finalize_manual_review_summary(
    adata_filtered,
    module='qc',
    workflow_name='notebook_qc_advanced',
    steps=qc_steps_executed,
    config=build_config_snapshot()['qc'],
    summary=qc_summary,
    save_dir=RESULTS_DIR / 'qc_review',
)
qc_specific_errors = validate_qc_review_summary(qc_review_summary)
if qc_specific_errors:
    print('QC review-summary warnings/errors:')
    for err in qc_specific_errors:
        print(f'  - {err}')

qc_validation = scl.qc.validate_qc_module_completeness(adata_filtered)
qc_compact = scl.qc.summarize_qc_review_summary(qc_review_summary)
print('\n---- QC Module Maturity ----')
print(f"QC module valid: {qc_validation['valid']}")
print(f"QC maturity: {qc_compact['maturity_status']}")
print(f"QC readiness: {qc_compact['readiness_status']} ({qc_compact['readiness_score']})")
print(f"QC retained cells: {qc_compact['final_cells']}")
if qc_validation['issues']:
    print('QC validation issues:')
    for issue in qc_validation['issues']:
        print(f'  - {issue}')
if qc_validation['warnings']:
    print('QC validation warnings:')
    for warning in qc_validation['warnings']:
        print(f'  ! {warning}')

from scLucid.utils import record_artifact
qc_output_path = DATA_DIR / 'Step1-sce_cleaned.h5ad'
record_artifact(
    adata_filtered,
    'qc',
    'qc_filtered_h5ad',
    str(qc_output_path),
    kind='h5ad',
    description='QC-filtered AnnData object',
)
adata_filtered.write(str(qc_output_path), compression='gzip')
print('\n---- QC Workflow Complete ----')
print(f'Saved: {qc_output_path}')
